### Main paper plot 3.5

Experiments on Fever to compare with the ground truth relationships.

Exp 3.5.1: F(S*(Q), Q) vs F(S(Q), Q) for S(Q) from all?/some? methods. In this case, X-axis is K upto 3.

Exp 3.5.2: MAP on ranking obtained by checking if S_1, S_2/S_1, S_3/S_2 and so on are within S^*. Table of MAP values 

In [ ]:
import torch
import pickle
import os
import numpy as np
import sys
from tqdm import tqdm

from omegaconf import OmegaConf
from src.utils import partial_chamfer_sim
from src.embedder import ColBERTEmbedder

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm

In [ ]:
from plot_utils import crop_pdf_with_fitz, crop_pdf_with_pdfcrop, crop_pdf_with_pypdf,\
    legend_labels, method_label_map, methods, legend_color_map, legend_marker_map

In [ ]:
methods = methods + ['gold']

In [ ]:
%load_ext autoreload
%autoreload 1

%aimport plot_utils

In [ ]:
datasets = ['fever']
dataset_name = datasets[0]
time_map, max_time_vals = plot_utils.get_time_data(datasets)

In [ ]:
# Load ground truth data for Science
from src.dataloader import get_dataloader

class DummyConfig:
    loader_type = 'beir'
    query_type = 'forum'
    dataset_name = dataset_name

dataloader = get_dataloader(DummyConfig())

In [ ]:
qrels = dataloader.get_qrels()

In [ ]:
corpus, _ = dataloader.get_data()

In [ ]:
list(corpus.keys())[:100]

In [ ]:
corpus, queries = dataloader.get_data()
# corpus: Dict[str, Dict]
keys = list(map(str, corpus.keys()))      # preserve a stable order
id2idx = {k: i for i, k in enumerate(keys)}
idx2id = keys                              # idx -> doc_id

In [ ]:
pos_indices = []
missing = []
for qrel_list in qrels.values():                    # qrels: Dict[doc_id -> 1]
    pos_indices_q = []
    missing_q = []

    for k in qrel_list:
        kk = str(k)
        if kk in id2idx:
            pos_indices_q.append(id2idx[kk])
        else:
            missing_q.append(kk)                # log/debug if any don't match

    pos_indices.append(pos_indices_q)
    missing.extend(missing_q)


In [ ]:
assert len(missing) == 0, f"Some qrels not found in corpus: {missing}"

In [ ]:
assert len(pos_indices) == len(queries)

In [ ]:
pos_indices[0]

To generate the scores for the gold sets, we have to use partial_chamfer_sim the same way we do for iid baselines.
That means we have to load the query and corpus embeddings.


In [ ]:
len(pos_indices)

In [ ]:
# Query indices with >= 2 relevant documents
multi_pos_qids = [i for i, pos in enumerate(pos_indices) if len(pos) >= 2]
len(multi_pos_qids)

In [ ]:
def get_S2_scores(scores, q_inds=None):
    "Returns the mean of the first two columns of the scores tensor. If q_inds is provided, only consider those indices."
    col_0 = []
    col_1 = []

    if q_inds is not None:
        for sc in scores[q_inds]:
            col_0.append(sc[0])
            col_1.append(sc[1])
    else:
        for sc in scores:
            col_0.append(sc[0])
            col_1.append(sc[1])
    
    print(f"Number of scores considered: {len(col_0)}")
    # No mean, just take the two columns
    S2_scores = torch.stack([torch.tensor(col_0), torch.tensor(col_1)], dim=1)
    return S2_scores

In [ ]:
# Load score data
score_map = {ds: {} for ds in datasets}
score_map_full = {ds: {} for ds in datasets}
ind_map = {ds: {} for ds in datasets}

for ds in datasets:
    for method in methods:
        if method == "gold":
            continue
        inds, scores = plot_utils.get_score_data(ds, method, k=10)
        print(f"Shapes for {ds}, {method}: inds: {inds.shape}, scores: {scores.shape}")
        assert inds.shape == scores.shape, f"Shape mismatch for {ds}, {method}: inds: {inds.shape}, scores: {scores.shape}"
        assert inds.shape[0] > 10
        S2_scores = get_S2_scores(scores, multi_pos_qids)
        score_map[ds][method] = S2_scores.mean(dim=0).numpy()
        score_map_full[ds][method] = S2_scores.numpy()
        ind_map[ds][method] = inds
        print(f"Mean scores for {ds}, {method}: {score_map[ds][method]}")

In [ ]:
def make_conf():
    # k=15 embedder.mode="disk" augment=False method='baseline' data.loader_type=lotte data.query_type=forum data.dataset_name=writing index=True overwrite_index=True
    sys.argv = ["", f"k=10", f"embedder.mode=disk", f"augment=False", f"method=baseline",
            f"data.loader_type=beir", f"data.query_type=forum", f"data.dataset_name={dataset_name}",
            "index=False", "overwrite_index=False"]
    print(sys.argv)
    cli_conf = OmegaConf.from_cli()
    main_conf = OmegaConf.load("configs/config.yaml")
    colbert_conf = OmegaConf.load("configs/colbert.yaml")

    conf = OmegaConf.merge(main_conf, colbert_conf, cli_conf)

    return conf

In [ ]:
corpus, _ = dataloader.get_data()
len(list(corpus))

In [ ]:
conf = make_conf()

In [ ]:
embedder = ColBERTEmbedder(conf.embedder)

In [ ]:
# Using pos_indices, we have to make a |Q| x max set size matrix of ground truth corpus items.
# Max set size means the largest number of relevant documents for any query.
# For padding purposes, we will repeat the last index in each query's list.

# Later we will index by pos_indices_desired
max_set_size = max(len(p) for p in pos_indices)
gt_indices = np.zeros((len(pos_indices), max_set_size), dtype=np.int32)
for i, p in enumerate(pos_indices):
    for j in range(max_set_size):
        if j < len(p):
            gt_indices[i, j] = p[j]
        else:
            gt_indices[i, j] = p[-1]
gt_indices = torch.from_numpy(gt_indices)

In [ ]:
gt_indices.shape

In [ ]:
embedder.embed_full_dataset(dataloader, mode=conf.embedder.mode)
qembs, qmasks = embedder.qembs, embedder.qmasks
gold_chamfer_scores = []

gt_indices = gt_indices.to(embedder.device)
print("Getting the corpus embeddings...")
corpus = embedder.get_corpus(gt_indices)
print("Done getting corpus embeddings.")
print(f"Corpus shape: {corpus.emb.shape}")

with open(f'./gold_set_corpus_{dataset_name}.pkl', 'wb') as f:
    pickle.dump(corpus, f)

In [ ]:
print("Computing chamfer scores for gold sets...")
gt_S2 = []
gold_chamfer_scores = []

for query_id in tqdm(range(len(queries)), desc="Computing chamfer scores for gold sets"):
    qemb = qembs[query_id:query_id+1]       # (1, Lq, d)
    qmask = qmasks[query_id:query_id+1]     # (1, Lq)
    
    cemb, cmask = corpus[
        (query_id * torch.ones(gt_indices.shape[1], dtype=torch.long)),
        torch.arange(gt_indices.shape[1], dtype=torch.long)
    ]

    out = partial_chamfer_sim(qembs[query_id][qmasks[query_id].bool()], cemb, cmask, device='cuda', bs=1024)
    out_sum = out.sum(dim=0)
    sorted_inds = torch.argsort(out_sum, descending=True)
    sorted_out = out[:, sorted_inds]
    gt_S2.append(gt_indices[query_id][sorted_inds])
    res = torch.cummax(sorted_out, dim=1)[0].sum(dim=0)
    gold_chamfer_scores.append(res)

In [ ]:
with open(f"./pickles/results/gold_chamfer_scores_{dataset_name}.pkl", "wb") as f:
    pickle.dump(gold_chamfer_scores, f)

In [ ]:
with open(f"./pickles/results/gold_chamfer_scores_{dataset_name}.pkl", "rb") as f:
    gold_chamfer_scores_try = pickle.load(f)

In [ ]:
gt_S2 = torch.vstack(gt_S2)
gt_S2.shape

In [ ]:
torch.stack(gold_chamfer_scores_try).shape

In [ ]:
dataset_name

In [ ]:
gold_chamfer_scores = torch.stack(gold_chamfer_scores).cpu().numpy()

In [ ]:
print("Before shape")
print(gold_chamfer_scores.shape)
print("After shape")
gold_chamfer_scores = gold_chamfer_scores[:, :2]
print(gold_chamfer_scores.shape)

In [ ]:
S2_scores_gold = get_S2_scores(gold_chamfer_scores, multi_pos_qids)
gc_mean = S2_scores_gold.mean(axis=0)
gc_mean.shape

In [ ]:
dms = ['disco - 1', 'submodlib ltl 0.5', 'exact greedy', 'MUVERA iid', 'submodlib lazy', 'WARP iid', 'submodlib stochastic 0.5', 'ColBERT iid']

In [ ]:
import pprint
# Get the scores at S_2
diffs = []
for method in dms:
    if method == "gold":
        continue
    else:
        score = score_map[dataset_name][method].tolist()[1]
        print(f"Raw S_2 score for {method}: {score} vs {gc_mean[1]}")
        score_diff = np.abs(score - gc_mean[1].numpy())
        diffs.append(score_diff)
    print(f"Method: {method}, Score difference at S_2: {score_diff}")

print(diffs)
srt_inds = np.argsort(diffs)
print("Method ranking:")
pprint.pprint([(dms[i], diffs[i]) for i in srt_inds])

In [ ]:
gold_chamfer_scores.shape

In [ ]:
# Get the scores at S_2: this is \sum_Q |F(S_2, Q) - F(S_2*, Q)| / |Q|, probably what we want.
mean_score_diffs = {method: 0.0 for method in dms if method != "gold"}
diffs = []
for method in dms:
    for idx, qid in enumerate(multi_pos_qids):
        if method == "gold":
            continue
        else:
            score = score_map_full[dataset_name][method][idx].tolist()[1]
            score_diff = np.abs(score - S2_scores_gold[idx][1])
            # print(f"QID: {qid}, Method: {method}, Score difference at S_2: {score_diff}")
            mean_score_diffs[method] += score_diff
    mean_score_diffs[method] /= len(multi_pos_qids)
    diffs.append(mean_score_diffs[method])
    print(f"Method: {method}, Mean score difference at S_2: {mean_score_diffs[method]}")

srt_inds = np.argsort(diffs)
print("Method ranking:")
pprint.pprint([(dms[i], diffs[i]) for i in srt_inds])

In [ ]:
# Get the scores at S_2: this is \sum_Q (1 - F(S_2, q) / F(S_2^*, q)) / |Q|.
mean_regret = {method: 0.0 for method in dms if method != "gold"}
diffs = []
for method in dms:
    for idx, qid in enumerate(multi_pos_qids):
        if method == "gold":
            continue
        else:
            score = score_map_full[dataset_name][method][idx].tolist()[1]
            score_diff = 1 - (score / S2_scores_gold[idx][1])
            # print(f"QID: {qid}, Method: {method}, Score difference at S_2: {score_diff}")
            mean_regret[method] += score_diff
    mean_regret[method] /= len(multi_pos_qids)
    diffs.append(mean_regret[method])
    print(f"Method: {method}, Mean score difference at S_2: {mean_regret[method]}")

srt_inds = np.argsort(diffs)
print("Method ranking:")
pprint.pprint([(dms[i], diffs[i]) for i in srt_inds])

In [ ]:
# Get avg coverage across all queries
for method in dms + ["gold"]:
    summ = 0
    for idx, qid in enumerate(multi_pos_qids):
        if method == "gold":
            score = S2_scores_gold[idx][1]
        else:
            score = score_map_full[dataset_name][method][idx].tolist()[1]
        summ += score
    avg_score = summ / len(multi_pos_qids)
    print(f"Method: {method}, Average score at S_2: {avg_score}")

### Experiment 3.5.2

In [ ]:
def compute_precision_acc(labels, predictions):
    sorted_predictions = sorted(((e, i) for i, e in enumerate(predictions)), reverse=True)
    precision = 0.0
    r = int(sum(labels))
    j = 1
    for i in range(len(sorted_predictions)):
        if labels[sorted_predictions[i][1]] == 1.0:
            precision += 1.0 * j / (i+1) / r
            j += 1

    return precision

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from sklearn.metrics import average_precision_score
from tqdm import tqdm

mrr_dict = {method: {} for method in methods if method != "gold"}
map_dict_1 = {method: {} for method in methods if method != "gold"}
map_dict_2 = {method: {} for method in methods if method != "gold"}
acc_dict = {method: {} for method in methods if method != "gold"}


# print(f"Gold set for query 543: {pos_indices[543]}")

for method in tqdm(methods, desc="Computing MAP for methods"):
    if method == 'gold':
        continue
    ap_list = []
    other_ap_list = []
    rr_list = []
    accs_list = []

    for q in tqdm(range(len(queries)), desc="Computing AP for queries"):
        if q not in multi_pos_qids:
            print(f"Skipping query {q} as it does not have multiple relevant documents.")
            continue
        gold_set = set(gt_S2[q].tolist())
        seen_so_far = set()
        y_gold = []
        count = 0
        y_score = []
        rr = 0
        acc = 0

        # print(f"Method {method}, Retrieved indices: {ind_map['hotpotqa'][method][q].tolist()}")
        # print(f"Method: {method}, Intersection: {gold_set.intersection(set(ind_map['hotpotqa'][method][q].tolist()))}")

        for element in ind_map[dataset_name][method][q]:
            element = element.item()
            if element in seen_so_far:
                break

            count = count + 1
            y_score.append(1/(count))
            if element in gold_set:
                y_gold.append(1)
                if rr == 0:
                    rr = 1/count
            else:
                y_gold.append(0)

            seen_so_far.add(element)

        rr_list.append(rr)
        # print(f"Element: {element}, y_gold: {y_gold}, y_score: {y_score}")

        ap = average_precision_score(y_gold, y_score)
        ap_list.append(ap)

        other_ap = compute_precision_acc(y_gold, y_score)
        other_ap_list.append(other_ap)

        acc = 1 if np.sum(y_gold) >= 2 else 0
        accs_list.append(acc)

    print(f"MAP for gold: {np.mean(ap_list)}")
    print(f"Other MAP for gold: {np.mean(other_ap_list)}")
    print(f"Acc for gold: {np.mean(accs_list)}")
    map_dict_1[method] = np.mean(ap_list)
    map_dict_2[method] = np.mean(other_ap_list)
    mrr_dict[method] = np.mean(rr_list)
    acc_dict[method] = np.mean(accs_list)

In [ ]:
from sklearn.metrics import average_precision_score
from tqdm import tqdm

mrr_dict_five = {method: {} for method in methods if method != "gold"}
map_dict_1_five = {method: {} for method in methods if method != "gold"}
map_dict_2_five = {method: {} for method in methods if method != "gold"}
acc_dict_five = {method: {} for method in methods if method != "gold"}


# print(f"Gold set for query 543: {pos_indices[543]}")

for method in tqdm(methods, desc="Computing MAP for methods"):
    if method == 'gold':
        continue
    ap_list = []
    other_ap_list = []
    rr_list = []
    accs_list = []

    for q in tqdm(range(len(queries)), desc="Computing AP for queries"):
        if q not in multi_pos_qids:
            print(f"Skipping query {q} as it does not have multiple relevant documents.")
            continue
        gold_set = set(gt_S2[q].tolist())
        seen_so_far = set()
        y_gold = []
        count = 0
        y_score = []
        rr = 0
        acc = 0

        # print(f"Method {method}, Retrieved indices: {ind_map['hotpotqa'][method][q].tolist()}")
        # print(f"Method: {method}, Intersection: {gold_set.intersection(set(ind_map['hotpotqa'][method][q].tolist()))}")

        for element in ind_map[dataset_name][method][q][:2]:
            element = element.item()
            if element in seen_so_far:
                break

            count = count + 1
            y_score.append(1/(count))
            if element in gold_set:
                y_gold.append(1)
                if rr == 0:
                    rr = 1/count
            else:
                y_gold.append(0)

            seen_so_far.add(element)

        rr_list.append(rr)
        # print(f"Element: {element}, y_gold: {y_gold}, y_score: {y_score}")

        ap = average_precision_score(y_gold, y_score)
        ap_list.append(ap)

        other_ap = compute_precision_acc(y_gold, y_score)
        other_ap_list.append(other_ap)

        acc = 1 if np.sum(y_gold) >= 2 else 0
        accs_list.append(acc)

    print(f"Method: {method}")
    print(f"MAP for gold: {np.mean(ap_list)}")
    print(f"Other MAP for gold: {np.mean(other_ap_list)}")
    print(f"Acc for gold: {np.mean(accs_list)}")
    map_dict_1_five[method] = np.mean(ap_list)
    map_dict_2_five[method] = np.mean(other_ap_list)
    mrr_dict_five[method] = np.mean(rr_list)
    acc_dict_five[method] = np.mean(accs_list)

In [ ]:
from sklearn.metrics import average_precision_score
from tqdm import tqdm

mrr_dict_five = {method: {} for method in methods if method != "gold"}
map_dict_1_five = {method: {} for method in methods if method != "gold"}
# map_dict_2_five = {method: {} for method in methods if method != "gold"}
# acc_dict_five = {method: {} for method in methods if method != "gold"}
last_rel_docs_dict = {method: {} for method in methods if method != "gold"}

freq_dict = {method: np.array([0] * 10) for method in methods if method != "gold"}
acc_dict = {method: np.array([0] * 3) for method in methods if method != "gold"}

queries_with_1_recall = {method: [] for method in methods if method != "gold"}

# print(f"Gold set for query 543: {pos_indices[543]}")

for method in tqdm(methods, desc="Computing MAP for methods"):
    if method == 'gold':
        continue
    ap_list = []
    other_ap_list = []
    rr_list = []
    accs_list = []
    last_rel_docs_list = []

    for q in tqdm(range(len(queries)), desc="Computing AP for queries"):
        if q not in multi_pos_qids:
            print(f"Skipping query {q} as it does not have multiple relevant documents.")
            continue
        gold_set = set(gt_S2[q].tolist())
        seen_so_far = set()
        y_gold = []
        count = 0
        y_score = []
        rr = 0
        acc = 0
        last_rel_doc = 11

        # print(f"Method {method}, Retrieved indices: {ind_map['hotpotqa'][method][q].tolist()}")
        # print(f"Method: {method}, Intersection: {gold_set.intersection(set(ind_map['hotpotqa'][method][q].tolist()))}")

        for element in ind_map[dataset_name][method][q]:
            element = element.item()
            if element in seen_so_far:
                break

            count = count + 1
            y_score.append(1/(count))
            if element in gold_set:
                y_gold.append(1)
                last_rel_doc = count if np.sum(y_gold) >= 2 else last_rel_doc
                if rr == 0:
                    rr = 1/count
            else:
                y_gold.append(0)

            seen_so_far.add(element)

        rr_list.append(rr)
        # print(f"Element: {element}, y_gold: {y_gold}, y_score: {y_score}")

        ap = average_precision_score(y_gold, y_score)
        ap_list.append(ap)

        other_ap = compute_precision_acc(y_gold, y_score)
        other_ap_list.append(other_ap)

        acc = 1 if np.sum(y_gold) >= 2 else 0
        accs_list.append(acc)

        last_rel_docs_list.append(last_rel_doc)
        freq_dict[method][min(last_rel_doc-1, 9)] += 1

    print(f"Method: {method}")
    print(f"MAP for gold: {np.mean(ap_list)}")
    # print(f"Other MAP for gold: {np.mean(other_ap_list)}")
    # print(f"Acc for gold: {np.mean(accs_list)}")
    map_dict_1_five[method] = np.mean(ap_list)
    # map_dict_2_five[method] = np.mean(other_ap_list)
    mrr_dict_five[method] = np.mean(rr_list)
    # acc_dict_five[method] = np.mean(accs_list)
    last_rel_docs_dict[method] = last_rel_docs_list

In [ ]:
import pprint
# pprint.pprint(map_dict, indent=2)
pprint.pprint(map_dict_1_five, indent=2)
pprint.pprint(map_dict_1, indent=2)
# pprint.pprint(map_dict_2, indent=2)

In [ ]:
pprint.pprint(acc_dict, indent=2)
pprint.pprint(acc_dict_five, indent=2)

In [ ]:
print(methods)

In [ ]:
import pprint
# print("Scipy MAP")
# pprint.pprint(map_dict_1, indent=2)

# print("Other MAP")
# pprint.pprint(map_dict_2, indent=2)

# print("MRR")
# pprint.pprint(mrr_dict, indent=2)

# print("Mean Acc")
# pprint.pprint(acc_dict, indent=2)
methods_test = ['submodlib lazy', 'submodlib stochastic 0.5', 'submodlib ltl 0.5', 'exact greedy', 'WARP iid', 'MUVERA iid', 'ColBERT iid', 'disco']

import pandas as pd
from collections import Counter

print("Last rel Docs - Bar Plot")

# Get all unique positions across all methods
all_positions = set()
for method in methods_test:
    if method == 'gold':
        continue
    all_positions.update(last_rel_docs_dict[method])

all_positions = sorted(list(all_positions))

# Create data for grouped bar plot
data_for_plot = {}
for method in methods_test:
    if method == 'gold':
        continue
    
    # Count frequency of each position for this method
    position_counts = Counter(last_rel_docs_dict[method])
    method_counts = [position_counts.get(pos, 0) for pos in all_positions]
    data_for_plot[method] = method_counts

# Create DataFrame for easier plotting
df = pd.DataFrame(data_for_plot, index=all_positions)

# Create grouped bar plot
plt.figure(figsize=(15, 8))
width = 0.8 / len(df.columns)  # Width of each bar
x_positions = range(len(all_positions))

for i, method in enumerate(df.columns):
    if 'disco' in method:
        color = 'black'
    elif 'ColBERT' in method and 'iid' in method:
        color = legend_color_map[method_label_map['ColBERT iid']]  # Use proper color mapping
    else:
        color = legend_color_map[method_label_map[method]]
    
    x_offset = [x + width * i for x in x_positions]
    plt.bar(x_offset, df[method], width, alpha=0.7, 
            label=method_label_map[method], color=color, edgecolor='black')

plt.xlabel('Position of Last Relevant Document Retrieved')
plt.ylabel('Number of Queries') 
plt.title('Frequency of Last Relevant Document Positions by Method')
plt.xticks([x + width * (len(df.columns) - 1) / 2 for x in x_positions], 
           all_positions, rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


# print("Last rel Docs")
# for method in methods_test:
#     if method == 'gold':
#         continue
#     if 'disco in method:
#         color = 'black'
#     elif 'ColBERT' in method and 'iid' in method:
#         color = 'white'
#     else:
#         color = legend_color_map[method_label_map[method]]

#     # Bar plot of frequency of last relevant document positions
    
#     plt.bar()
#     plt.hist(last_rel_docs_dict[method], bins=10, alpha=0.6, label=method, color=color,
#              histtype='stepfilled', edgecolor='black')
# plt.xlabel('Position of Last Relevant Document Retrieved')
# plt.ylabel('Number of Queries')
# plt.title('Histogram of Last Relevant Document Positions by Method')
# plt.legend()
# plt.show()

In [ ]:
import pandas as pd
from collections import Counter

def histogram_processing(methods, last_rel_docs_dict):
    # Get all unique positions across all methods
    all_positions = set()
    for method in methods_test:
        if method == 'gold':
            continue
        all_positions.update(last_rel_docs_dict[method])

    all_positions = sorted(list(all_positions))

    # Create data for grouped bar plot
    data_for_plot = {}
    for method in methods_test:
        if method == 'gold':
            continue

        # Count frequency of each position for this method
        position_counts = Counter(last_rel_docs_dict[method])
        method_counts = [position_counts.get(pos, 0) for pos in all_positions]
        data_for_plot[method] = method_counts

    # Create DataFrame for easier plotting
    df = pd.DataFrame(data_for_plot, index=all_positions)

    # keep only positions <= 5
    positions_to_show = [p for p in all_positions if p <= 5]

    # slice the DataFrame
    df_small = df.loc[positions_to_show]

    return df_small, positions_to_show

In [ ]:
import pandas as pd
from collections import Counter


def plot_histogram(last_rel_docs_dict):
    methods_test = ['submodlib lazy', 'submodlib stochastic 0.5', 'submodlib ltl 0.5', 'exact greedy', 'WARP iid', 'MUVERA iid', 'ColBERT iid', 'disco - 1', 'gold']

    print("Last rel Docs - Bar Plot")

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        'figure.dpi': 300,
        # 'lines.markersize': 12
    })

    df_small, positions_to_show = histogram_processing(methods_test, last_rel_docs_dict)
    fig, ax = plt.subplots(figsize=(15, 8))
    # plt.figure(figsize=(15, 8))
    width = 0.8 / len(df_small.columns)
    x_positions = range(len(positions_to_show))

    for i, method in enumerate(df_small.columns):
        if 'disco' in method:
            color = 'black'
        elif 'ColBERT' in method and 'iid' in method:
            color = plot_utils.legend_color_map[method_label_map['ColBERT iid']]  # Use proper color mapping
        else:
            color = plot_utils.legend_color_map[method_label_map[method]]
        x_offset = [x + width * i for x in x_positions]
        plt.bar(x_offset, df_small[method], width, alpha=0.7,
                label=plot_utils.method_label_map[method], color=color, edgecolor='black',
                align='center')

    ax.set_xlabel(r'\textbf{Rank of Last Relevant Item Retrieved}', fontsize=48)
    ax.set_ylabel(r'\textbf{Number of Queries}', fontsize=48)
    # plt.title('Frequency of Last Relevant Document Positions by Method')
    plt.xticks([x + width * (len(df_small.columns) - 1) / 2 for x in x_positions],
            positions_to_show) #, rotation=45)
    # plt.xticks([2, 3, 4, 5])

    # Explicitly set tick label sizes
    ax.tick_params(axis='x', labelsize=60)  # Smaller X-axis tick labels
    ax.tick_params(axis='y', labelsize=60)  # Smaller Y-axis tick labels

    # Get current x-tick values and format them with LaTeX bold
    xticks = ax.get_xticks()
    # xticklabels = [fr'$\mathbf{{{int(v)}}}$' for v in xticks]
    # xticklabels = [r'$\mathbf{0}$', r'$\mathbf{2}$', r'$\mathbf{3}$', r'$\mathbf{4}$', r'$\mathbf{5}$']
    # ax.set_xticklabels(xticklabels)

    # Get current y-tick values and format them with LaTeX bold
    yticks = ax.get_yticks()
    yticklabels = [fr'$\mathbf{{{int(v)}}}$' for v in yticks]
    ax.set_yticklabels(yticklabels)

    # plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'./notebooks/plots/{dataset_name}_plot3_5_histogram.pdf', bbox_inches='tight', dpi=300)
    plt.show()


In [ ]:
plot_histogram(last_rel_docs_dict)

In [ ]:
import pandas as pd
from collections import Counter


def plot_histogram_legend(last_rel_docs_dict, filename, ncol=3, auto_crop=True):
    methods_test = ['disco - 1', 'submodlib ltl 0.5', 'exact greedy', 'MUVERA iid', 'submodlib lazy', 'WARP iid', 'submodlib stochastic 0.5', 'ColBERT iid']
    print("Last rel Docs - Bar Plot")

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        'figure.dpi': 300,
        'lines.markersize': 12
    })

    df_small, positions_to_show = histogram_processing(methods_test, last_rel_docs_dict)
    fig, ax = plt.subplots(figsize=(15, 8))
    # plt.figure(figsize=(15, 8))
    width = 0.8 / len(df_small.columns)
    x_positions = range(len(positions_to_show))

    for i, method in enumerate(df_small.columns):
        if 'disco' in method:
            color = 'black'
        elif 'ColBERT' in method and 'iid' in method:
            color = plot_utils.legend_color_map[method_label_map['ColBERT iid']]  # Use proper color mapping
        else:
            color = plot_utils.legend_color_map[method_label_map[method]]
        x_offset = [x + width * i for x in x_positions]
        plt.bar([], [], width, alpha=0.7,
                label=plot_utils.method_label_map[method], color=color, edgecolor='black')

    # Create legend
    legend = ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=ncol,
        fontsize=25,
        frameon=False
    )

    # Save the figure
    output_path = f'./notebooks/plots/{filename}.pdf'
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()
    
    # Auto-crop the PDF if requested
    if auto_crop:
        # Try methods in order of preference
        cropped_path = crop_pdf_with_pdfcrop(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_fitz(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_pypdf(output_path)
        
        if cropped_path:
            # Replace original with cropped version
            os.rename(cropped_path, output_path)
            print(f"PDF automatically cropped: {output_path}")
        else:
            print("Auto-cropping failed, using matplotlib's bbox_inches='tight' only")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def legend_only(methods_order, legend_color_map, method_label_map, filename, ncol=3):
    # Build labels & colors
    labels = [method_label_map[m] for m in methods_order]
    def color_for(m):
        if 'disco' in m:
            return 'black'
        if 'ColBERT' in m and 'iid' in m:
            return plot_utils.legend_color_map[method_label_map['ColBERT iid']]
        return plot_utils.legend_color_map[method_label_map[m]]

    colors = [color_for(m) for m in methods_order]

    # Proxy handles for legend
    handles = [Patch(facecolor=c, edgecolor='black', label=lbl)
               for c, lbl in zip(colors, labels)]

    fig = plt.figure(figsize=(8, 2.2), dpi=300)
    # no axes at all
    leg = fig.legend(handles=handles,
                     loc="center", ncol=ncol, frameon=False, fontsize=30)
    fig.savefig(f'./notebooks/plots/{filename}.pdf',
                bbox_inches='tight', pad_inches=0.1)
    plt.show(fig)
    # plt.close(fig)


In [ ]:
dms = ['disco - 1', 'submodlib ltl 0.5', 'exact greedy', 'MUVERA iid', 'submodlib lazy', 'WARP iid', 'submodlib stochastic 0.5', 'ColBERT iid']

legend_only(dms, legend_color_map=plot_utils.legend_color_map,
            method_label_map=plot_utils.method_label_map, filename='plot_3_5_legend', ncol=4)


In [ ]:
for dataset in datasets:
    for method in methods:
        if method == "gold":
            scores = gc_mean
        else:
            scores = score_map[dataset][method].tolist()[:2]

        print(f"Method: {method}, dataset: {dataset}, scores: {scores}")

### QRels avg and max set size

In [ ]:
# Check msmarco specifically
class DummyConfig:
    loader_type = 'beir'
    dataset_name = 'msmarco'

dloader = get_dataloader(DummyConfig())
qrels_msmarco = dloader.get_qrels()
qrels_msmarco[0]

In [ ]:
qrels_msmarco['19335']

In [ ]:
# Check pooled specifically
class DummyConfig:
    loader_type = 'lotte'
    query_type = 'forum'
    dataset_name = 'pooled'

dloader = get_dataloader(DummyConfig())
qrels_pooled = dloader.get_qrels()

In [ ]:
qrels_pooled['0']

In [ ]:
# Check msmarco specifically
class DummyConfig:
    loader_type = 'beir'
    dataset_name = 'fever'

dloader = get_dataloader(DummyConfig())
qrels_fever = dloader.get_qrels()

In [ ]:
qrels_fever['163803']

In [ ]:
for dataset in ['pooled', 'science', 'technology']:
    class DummyConfig:
        loader_type = 'lotte'
        query_type = 'forum'
        dataset_name = dataset

    dloader = get_dataloader(DummyConfig())
    qrels = dloader.get_qrels()
    print(f"Dataset: {dataset}, Number of queries with at least one relevant doc: {len(qrels)}")
    print(f"Avg relevant docs per query: {np.mean([len(v) for v in qrels.values()])}")
    print(f"Maximum relevant docs for a single query: {max(len([val for val in v if int(val) >= 1]) for v in qrels.values())}")

for dataset in ['msmarco', 'hotpotqa']:
    class DummyConfig:
        loader_type = 'beir'
        dataset_name = dataset

    dloader = get_dataloader(DummyConfig())
    qrels = dloader.get_qrels()
    print(f"Dataset: {dataset}, Number of queries with at least one relevant doc: {len(qrels)}")
    print(f"Avg relevant docs per query: {np.mean([len([val for val in v.values() if val >= 1]) for v in qrels.values()])}")
    print(f"Maximum value of relevance label: {max([max([int(val) for val in v.values()]) for v in qrels.values()])}")
    print(f"Maximum relevant docs for a single query: {max(len([val for val in v.values() if val >= 1]) for v in qrels.values())}")

In [ ]:
dataset = 'fever'
class DummyConfig:
    loader_type = 'beir'
    dataset_name = dataset

dloader = get_dataloader(DummyConfig())
qrels = dloader.get_qrels()
print(f"Dataset: {dataset}, Number of queries with at least one relevant doc: {len(qrels)}")
print(f"Avg relevant docs per query: {np.mean([len([val for val in v.values() if val >= 1]) for v in qrels.values()])}")
print(f"Maximum value of relevance label: {max([max([int(val) for val in v.values()]) for v in qrels.values()])}")
print(f"Maximum relevant docs for a single query: {max(len([val for val in v.values() if val >= 1]) for v in qrels.values())}")


In [ ]:
print(f"Number of queries with exactly one relevant doc: {len([1 for v in qrels.values() if len([val for val in v.values() if val >= 1]) == 1])}")

In [ ]:
print(f"Number of queries with two relevant doc: {len([1 for v in qrels.values() if len([val for val in v.values() if val >= 1]) == 2])}")

In [ ]:
print(f"Number of queries with atleast two relevant doc: {len([1 for v in qrels.values() if len([val for val in v.values() if val >= 1]) >= 2])}")

In [ ]:
print(f"Number of queries with three relevant doc: {len([1 for v in qrels.values() if len([val for val in v.values() if val >= 1]) == 3])}")

In [ ]:
dataset = 'hotpotqa'
class DummyConfig:
    loader_type = 'beir'
    dataset_name = dataset

dloader = get_dataloader(DummyConfig())
qrels = dloader.get_qrels()
print(f"Dataset: {dataset}, Number of queries with at least one relevant doc: {len(qrels)}")
print(f"Avg relevant docs per query: {np.mean([len([val for val in v.values() if val >= 1]) for v in qrels.values()])}")
print(f"Maximum value of relevance label: {max([max([int(val) for val in v.values()]) for v in qrels.values()])}")
print(f"Maximum relevant docs for a single query: {max(len([val for val in v.values() if val >= 1]) for v in qrels.values())}")
